# BIGFix: text-to-image demo

Minimal text-to-image inference with the [BigFIX xlarge/512 checkpoint](https://huggingface.co/llvictorll/BigFIX) from Hugging Face. A GPU is required (Colab: *Runtime > Change runtime type > GPU*).

## 1. Setup

On Colab this clones the repository and installs it. Locally, `pip install -e .` from the repository root first; the notebook can then be run from anywhere.

In [ ]:
import os
import sys

if "google.colab" in sys.modules and not os.path.exists("BigFix"):
    !git clone https://github.com/valeoai/BigFix.git
    %cd BigFix
    !pip install -q -e .

## 2. Load the model

The transformer (7.5 GB) and the VQGAN are downloaded into `./saved_networks/` on the first call, and the FLAN-T5 text encoder (about 11 GB of files) is fetched from the Hugging Face Hub, so the first run takes a few minutes.

The model needs about 8 GB of GPU memory (2.3 GB transformer, 4.6 GB text encoder, plus the VQGAN). The transformer checkpoint also stores the optimizer state (5 GB); it is memory-mapped, so only the 2.3 GB of weights are read into CPU RAM.

In [ ]:
from bigfix import MaskGITPipeline

pipe = MaskGITPipeline.from_pretrained(
    repo_id="llvictorll/BigFIX",
    vit_filename="BigFix_XLarge_aplha02_res512_MiroFT.pth",
    vqgan_filename="vq_ds16_t2i.pt",
    config="xlarge_txt2img_gpic.yaml",
    img_size=512,
).to("cuda")

## 3. Prompts

`PROMPTS` is the list the trainer renders during training to monitor samples (`bigfix/trainer/txt_trainer.py`). Pick from it by index and/or write your own.

In [ ]:
PROMPTS = [
    "An elephant riding a vintage bicycle through a bustling city street, hyper-realistic detail, 4k UHD.",
    "A joyful man walking his loyal dog through a picturesque park at sunset, cinematic lighting.",
    "A mystical fox in an enchanted forest, glowing flora, and soft mist, rendered in Unreal Engine.",
    "A sleek airplane soaring above the clouds during a vibrant sunset, with a stunning view of the horizon.",
    "An underwater paradise teeming with colorful, exotic fish swimming through coral reefs in crystal-clear ocean water.",
    "A Pikachu enjoying an elegant five-star meal with a breathtaking view of the Eiffel Tower, during a golden sunset.",
    "A towering mecha robot overlooking a vibrant favela, painted in bold, abstract expressionist style.",
    "An insect-robot chef expertly crafting a gourmet meal in a high-tech futuristic kitchen, intricate details.",
    "A cozy wooden cabin perched on a snowy mountain peak, glowing warmly in the night, styled like a classic Disney movie, featured on ArtStation.",
    "A cute little matte low poly isometric cherry blossom forest island, waterfalls, lighting, soft shadows, trending on Artstation, 3D render, Monument Valley, Fez video game.",
    "A shanty version of Tokyo, new rustic style, bold colors with all colors palette, video game, Genshin, tribe, fantasy, Overwatch.",
    "A cozy gingerbread house nestled in a dusting of powdered sugar snow adorned with vibrant candy canes and shimmering gumdrops.",
    "A teddy bear wearing a blue ribbon taking a selfie in a small boat in the center of a lake.",
    "A pirate ship trapped in a cosmic maelstrom nebula rendered in cosmic beach whirlpool engine.",
    "Volumetric lighting, spectacular ambient lights, light pollution, cinematic atmosphere, Art Nouveau style illustration art, artwork by SenseiJaye, intricate detail.",
    "A blue jay stops on the top of a helmet of a Japanese samurai, background with sakura tree.",
    "A shark flying in the desert.",
    "A tiny astronaut hatching from an transparent egg on the moon.",
    "An old-world galleon navigating through turbulent ocean waves under a stormy sky lit by flashes of lightning.",
    "An oil painting of rain at a traditional Chinese town.",
    "Portrait photo of an Asian old warrior chief, tribal panther makeup, blue on red, side profile, looking away, serious eyes, 50mm portrait photography, hard rim lighting.",
    "A female character with long, flowing hair that appears to be made of ethereal, swirling patterns resembling the Northern Lights or Aurora Borealis. The background is dominated by deep blues and purples, creating a mysterious and dramatic atmosphere. The character's face is serene, with pale skin and striking features. She wears a dark-colored outfit with subtle patterns. The overall style of the artwork is reminiscent of fantasy or supernatural genres",
    "Digital art, portrait of an anthropomorphic roaring Tiger warrior with full armor, close up in the middle of a battle, behind him there is a banner with the text 'Open Source'.",
    "Photo of a dog and a cat both standing on a red box, with a blue ball in the middle with a parrot standing on top of the ball. The box has the text 'Victory' written on it",
    "Selfie photo of a wizard with long beard and purple robes, he is apparently in the middle of Tokyo. Probably taken from a phone.",
    "A vibrant street wall covered in colorful graffiti, the centerpiece spells 'HALTON', in a storm of colors.",
    "Photo of a young woman with long, wavy brown hair tied in a bun and glasses. She has a fair complexion and is wearing subtle makeup, emphasizing her eyes and lips. She is dressed in a black top. The background appears to be an urban setting with a building facade, and the sunlight casts a warm glow on her face.",
    "Anime art of a steampunk inventor in their workshop, surrounded by gears, gadgets, and steam. He is holding a blue potion and a red potion, one in each hand",
    "Photo of picturesque scene of a road surrounded by lush green trees and shrubs. The road is wide and smooth, leading into the distance. On the right side of the road, there's a blue sports car parked with the license plate spelling 'SD32B'. The sky above is partly cloudy, suggesting a pleasant day. The trees have a mix of green and brown foliage. There are no people visible in the image. The overall composition is balanced, with the car serving as a focal point.",
    "Photo of a young man in a black suit, white shirt, and black tie. He has a neatly styled haircut and is looking directly at the camera with a neutral expression. The background consists of a textured wall with horizontal lines. The photograph is in black and white, emphasizing contrasts and shadows. The man appears to be in his late twenties or early thirties, with fair skin and short, dark hair.",
    "Photo of a woman on the beach, shot from above. She is facing the sea, while wearing a white dress. She has long blonde hair",
]

# Appended to every prompt when `quality_suffix=True` below (the suffix `sample_test` uses for its evaluation grid).
QUALITY_SUFFIX = (". The image is ultra-detailed, with cinematic lighting, high clarity, intricate textures, "
                  "and beautifully composed. Emphasize depth and dynamic shadows for a lifelike, visually captivating result.")

for i, p in enumerate(PROMPTS):
    print(f"{i:2d}  {p[:110]}{'...' if len(p) > 110 else ''}")

In [ ]:
prompt_ids = [17, 20, 24]          # indices into PROMPTS above ([] to use only your own prompts)
custom_prompts = [                 # your own prompts, generated after the ones picked above
    # "A red fox reading a book in a cozy library, warm light.",
]
negative_prompt = None             # e.g. "blurry, low quality, watermark" (steers away from it through CFG)
quality_suffix = False             # True: append QUALITY_SUFFIX to every prompt

prompts = [PROMPTS[i] for i in prompt_ids] + custom_prompts
if quality_suffix:
    prompts = [p + QUALITY_SUFFIX for p in prompts]
print(f"{len(prompts)} prompt(s) selected")

## 4. Sampling hyperparameters

Every entry of `sampling` is forwarded to the pipeline call; the values below are the defaults of `xlarge_txt2img_gpic.yaml`, except `scheduler`, `lin_cfg_warmup` and `start_correction`, which are the sampler's own defaults.

| Parameter | Effect |
|---|---|
| `num_inference_steps` | Number of Halton sampling steps. More steps means fewer tokens revealed per step, usually better quality but slower. |
| `guidance_scale` | Classifier-free guidance weight. Higher follows the prompt more closely, at the cost of diversity and, when too high, saturated or artefacted images. `0` disables guidance (twice as fast). |
| `sm_temp_min`, `sm_temp_max` | Softmax temperature at the first and last step, linearly interpolated in between. It **multiplies** the logits, so higher is sharper and less random. |
| `top_k` | Sample only among the `k` most likely tokens (`-1` disables). |
| `top_p` | Nucleus sampling: sample among the smallest set of tokens whose probability sums to `top_p` (`1.0` disables). |
| `scheduler` | How the number of revealed tokens grows over the steps: `arccos` (default), `linear`, `sqrt`, `square` or `cos`. |
| `randomize` | Randomly rotate the Halton order for each image, so samples of the same prompt reveal tokens in different orders. |
| `lin_cfg_warmup` | Ramp the guidance linearly from 0 up to `guidance_scale` instead of using it at full strength from the first step. |
| `start_correction` | During the last `start_correction` steps every token revealed so far is re-sampled, which lets the model fix earlier mistakes. |

In [ ]:
sampling = dict(
    num_inference_steps=32,
    guidance_scale=5.0,
    sm_temp_min=1.0,
    sm_temp_max=1.0,
    top_k=-1,
    top_p=0.8,
    scheduler="arccos",
    randomize=False,
    lin_cfg_warmup=False,
    start_correction=32,
)

## 5. Generate

`batch_size` is the number of images sampled at once: lower it if you run out of GPU memory (classifier-free guidance doubles the effective batch). Each batch is seeded with `seed + batch_index`, so a given `seed` and `batch_size` always give the same images.

In [ ]:
import math
import textwrap

import matplotlib.pyplot as plt


def show(images, captions=None, ncols=None, size=4):
    """ Display PIL images in a grid, with (shortened) captions as titles. """
    ncols = ncols or min(len(images), 4)
    nrows = math.ceil(len(images) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, img in enumerate(images):
        axes.flat[k].imshow(img)
        if captions:
            axes.flat[k].set_title(textwrap.shorten(captions[k], 70, placeholder="..."), fontsize=8)
    plt.tight_layout()
    plt.show()


def generate(prompts, num_images_per_prompt=1, batch_size=4, seed=42, negative_prompt=None, **sampling):
    """ Returns (images, captions), one entry per (prompt, sample) pair, in prompt order. """
    flat = [p for p in prompts for _ in range(num_images_per_prompt)]
    images = []
    for b, start in enumerate(range(0, len(flat), batch_size)):
        out = pipe(prompt=flat[start:start + batch_size], negative_prompt=negative_prompt,
                   seed=seed + b, verbose=True, **sampling)
        images += out.images
    return images, flat

In [ ]:
num_images_per_prompt = 1
seed = 42
batch_size = 4

images, captions = generate(prompts, num_images_per_prompt=num_images_per_prompt, batch_size=batch_size,
                            seed=seed, negative_prompt=negative_prompt, **sampling)
show(images, captions)

## 6. Compare hyperparameters

Sweep one parameter over several values for the same prompt and seed, everything else being taken from `sampling`. Only a single image is sampled per value, so the runs are directly comparable.

In [ ]:
def sweep(prompt, param, values, seed=42, **overrides):
    """ One image per value of `param`, same prompt and seed, the rest of `sampling` unchanged. """
    settings = {**sampling, **overrides}
    images = []
    for v in values:
        images += pipe(prompt=prompt, seed=seed, **{**settings, param: v}).images
    show(images, [f"{param}={v}" for v in values])


sweep(prompts[0], "guidance_scale", [0.0, 2.0, 5.0, 8.0])

In [ ]:
# More sweeps to try:
# sweep(prompts[0], "num_inference_steps", [8, 16, 32, 64])
# sweep(prompts[0], "top_p", [0.5, 0.8, 0.95, 1.0])
# sweep(prompts[0], "sm_temp_max", [0.8, 1.0, 1.2, 1.5])
# sweep(prompts[0], "scheduler", ["arccos", "cos", "linear", "sqrt", "square"])
# sweep(prompts[0], "start_correction", [0, 8, 16, 32], num_inference_steps=64)

## 7. Save

In [ ]:
import os
import re

os.makedirs("outputs", exist_ok=True)
for k, (img, cap) in enumerate(zip(images, captions)):
    slug = re.sub(r"[^a-z0-9]+", "_", cap.lower()).strip("_")[:50]
    img.save(f"outputs/{k:02d}_{slug}.png")
print(f"Saved {len(images)} image(s) in ./outputs/")